In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
import sys
from uuid import UUID
from typing import List, Optional
from sqlalchemy import create_engine, text, select
from sqlalchemy.orm import Session
from dotenv import load_dotenv
import alite_backend
from alite_backend.db.db_session import SessionLocal
from alite_backend.db.models import Lemma, Lexeme, GramProp, WordForm, Base
from alite_backend.words.pipeline import feed_data
from alite_backend.words.lookup import LookupFDAPI as lfa
from alite_backend.words.process import ReturnedLemmaProcessor as rlp
from alite_backend.db.schemas import (
    LemmasRecord,
    GramPropsRecord,
    LexiconRecord,
    DefinitionsRecord,
    DefExamplesRecord,
    PronunciationsRecord,
    VerbPairsRecord,
    ProcessedPayload,
)

In [62]:
load_dotenv()
APP_DIR = os.getenv("APP_DIR")
INIT_DB_LOC = APP_DIR + "backend/src/alite_backend/db/init_db.sql"

In [ ]:
podkhod = {
    "lemmas": [
        {
            "clean_lemma": "подход",
            "accent_lemma": "подхо́д",
            "pos": 5,
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
        }
    ],
    "gram_props": [
        {
            "temp_form_id": "c14bba6a-a05e-4ba6-9410-9c84d253268e",
            "prop_name": "canonical",
        },
        {
            "temp_form_id": "c14bba6a-a05e-4ba6-9410-9c84d253268e",
            "prop_name": "inanimate",
        },
        {
            "temp_form_id": "c14bba6a-a05e-4ba6-9410-9c84d253268e",
            "prop_name": "masculine",
        },
        {
            "temp_form_id": "95dedc5a-e0da-4aa6-9a0b-14196de2cf61",
            "prop_name": "romanization",
        },
        {
            "temp_form_id": "101212af-2ee7-4107-aecd-a0cec87edd05",
            "prop_name": "genitive",
        },
        {
            "temp_form_id": "29b06da4-6029-4e50-a717-95196a088e51",
            "prop_name": "nominative",
        },
        {"temp_form_id": "29b06da4-6029-4e50-a717-95196a088e51", "prop_name": "plural"},
        {
            "temp_form_id": "891fc605-8ac6-4738-8868-9faef7975bda",
            "prop_name": "genitive",
        },
        {"temp_form_id": "891fc605-8ac6-4738-8868-9faef7975bda", "prop_name": "plural"},
        {
            "temp_form_id": "eb21d97d-0f73-4696-9c67-72c1d0fd476b",
            "prop_name": "table-tags",
        },
        {
            "temp_form_id": "da1a7394-904a-4042-b614-2242ff28e782",
            "prop_name": "inflection-template",
        },
        {"temp_form_id": "03381e8f-da62-40cf-be09-e2071e5a3180", "prop_name": "class"},
        {"temp_form_id": "296e0540-8271-4711-9ad3-e5570a76f6b9", "prop_name": "class"},
        {
            "temp_form_id": "1dc13fd9-2a16-418e-82a9-99bf910eea18",
            "prop_name": "nominative",
        },
        {
            "temp_form_id": "1dc13fd9-2a16-418e-82a9-99bf910eea18",
            "prop_name": "singular",
        },
        {
            "temp_form_id": "707fd4cd-48d7-46a3-ad97-51c846e3d357",
            "prop_name": "nominative",
        },
        {"temp_form_id": "707fd4cd-48d7-46a3-ad97-51c846e3d357", "prop_name": "plural"},
        {
            "temp_form_id": "ab50d0f1-7ff7-47aa-a1d3-8f3bb75ed5a1",
            "prop_name": "genitive",
        },
        {
            "temp_form_id": "ab50d0f1-7ff7-47aa-a1d3-8f3bb75ed5a1",
            "prop_name": "singular",
        },
        {
            "temp_form_id": "f20ff19c-8869-4aa4-83d0-313e2756dd37",
            "prop_name": "genitive",
        },
        {"temp_form_id": "f20ff19c-8869-4aa4-83d0-313e2756dd37", "prop_name": "plural"},
        {"temp_form_id": "95707190-701a-40f0-ad68-1c8c485bebd9", "prop_name": "dative"},
        {
            "temp_form_id": "95707190-701a-40f0-ad68-1c8c485bebd9",
            "prop_name": "singular",
        },
        {"temp_form_id": "33e10e53-50ed-4e27-8506-3d8b63d86095", "prop_name": "dative"},
        {"temp_form_id": "33e10e53-50ed-4e27-8506-3d8b63d86095", "prop_name": "plural"},
        {
            "temp_form_id": "f976f0fa-691e-403f-9939-f2d53ed8b505",
            "prop_name": "accusative",
        },
        {
            "temp_form_id": "f976f0fa-691e-403f-9939-f2d53ed8b505",
            "prop_name": "singular",
        },
        {
            "temp_form_id": "ab5475a8-c205-4b44-95ac-5577d89646c0",
            "prop_name": "accusative",
        },
        {"temp_form_id": "ab5475a8-c205-4b44-95ac-5577d89646c0", "prop_name": "plural"},
        {
            "temp_form_id": "bf9b776d-6abb-4924-b2bb-1b3bdf7c78a6",
            "prop_name": "instrumental",
        },
        {
            "temp_form_id": "bf9b776d-6abb-4924-b2bb-1b3bdf7c78a6",
            "prop_name": "singular",
        },
        {
            "temp_form_id": "b2d46d19-2e02-4837-8324-cdbb11abeaa3",
            "prop_name": "instrumental",
        },
        {"temp_form_id": "b2d46d19-2e02-4837-8324-cdbb11abeaa3", "prop_name": "plural"},
        {
            "temp_form_id": "e658f9ab-a60a-4b0a-ac73-e1172418829b",
            "prop_name": "prepositional",
        },
        {
            "temp_form_id": "e658f9ab-a60a-4b0a-ac73-e1172418829b",
            "prop_name": "singular",
        },
        {"temp_form_id": "bce0d6ef-8d13-4f2c-9f56-1717d56cfd1a", "prop_name": "plural"},
        {
            "temp_form_id": "bce0d6ef-8d13-4f2c-9f56-1717d56cfd1a",
            "prop_name": "prepositional",
        },
        {
            "temp_form_id": "29fe398d-45e6-404f-8f59-c3a840f16238",
            "prop_name": "alternative",
        },
    ],
    "lexicon": [
        {
            "temp_form_id": "c14bba6a-a05e-4ba6-9410-9c84d253268e",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "подхо́д",
        },
        {
            "temp_form_id": "95dedc5a-e0da-4aa6-9a0b-14196de2cf61",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "podxód",
        },
        {
            "temp_form_id": "101212af-2ee7-4107-aecd-a0cec87edd05",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "подхо́да",
        },
        {
            "temp_form_id": "29b06da4-6029-4e50-a717-95196a088e51",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "подхо́ды",
        },
        {
            "temp_form_id": "891fc605-8ac6-4738-8868-9faef7975bda",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "подхо́дов",
        },
        {
            "temp_form_id": "eb21d97d-0f73-4696-9c67-72c1d0fd476b",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "no-table-tags",
        },
        {
            "temp_form_id": "da1a7394-904a-4042-b614-2242ff28e782",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "ru-noun-table",
        },
        {
            "temp_form_id": "03381e8f-da62-40cf-be09-e2071e5a3180",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "hard-stem",
        },
        {
            "temp_form_id": "296e0540-8271-4711-9ad3-e5570a76f6b9",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "accent-a",
        },
        {
            "temp_form_id": "1dc13fd9-2a16-418e-82a9-99bf910eea18",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "подхо́д",
        },
        {
            "temp_form_id": "707fd4cd-48d7-46a3-ad97-51c846e3d357",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "подхо́ды",
        },
        {
            "temp_form_id": "ab50d0f1-7ff7-47aa-a1d3-8f3bb75ed5a1",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "подхо́да",
        },
        {
            "temp_form_id": "f20ff19c-8869-4aa4-83d0-313e2756dd37",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "подхо́дов",
        },
        {
            "temp_form_id": "95707190-701a-40f0-ad68-1c8c485bebd9",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "подхо́ду",
        },
        {
            "temp_form_id": "33e10e53-50ed-4e27-8506-3d8b63d86095",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "подхо́дам",
        },
        {
            "temp_form_id": "f976f0fa-691e-403f-9939-f2d53ed8b505",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "подхо́д",
        },
        {
            "temp_form_id": "ab5475a8-c205-4b44-95ac-5577d89646c0",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "подхо́ды",
        },
        {
            "temp_form_id": "bf9b776d-6abb-4924-b2bb-1b3bdf7c78a6",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "подхо́дом",
        },
        {
            "temp_form_id": "b2d46d19-2e02-4837-8324-cdbb11abeaa3",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "подхо́дами",
        },
        {
            "temp_form_id": "e658f9ab-a60a-4b0a-ac73-e1172418829b",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "подхо́де",
        },
        {
            "temp_form_id": "bce0d6ef-8d13-4f2c-9f56-1717d56cfd1a",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "подхо́дах",
        },
        {
            "temp_form_id": "29fe398d-45e6-404f-8f59-c3a840f16238",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "form": "подхо́дъ",
        },
    ],
    "definitions": [
        {
            "temp_def_id": "1ed53952-9f77-4e78-a5e9-612457993516",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "def_text": "approach",
            "tags": [],
        },
        {
            "temp_def_id": "a9697dc3-dd26-4f18-98b1-43d3daeaeabb",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "def_text": "point of view",
            "tags": [],
        },
        {
            "temp_def_id": "77df2f11-95a7-4d58-911c-b677582bc8f9",
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "def_text": "set (a certain amount of repetitions of a physical exercise)",
            "tags": [],
        },
    ],
    "def_sentences": [
        {
            "temp_def_id": "a9697dc3-dd26-4f18-98b1-43d3daeaeabb",
            "def_sentence": "Предвари́тельный план де́йствий предполага́ет децентрализо́ванный подхо́д к организа́ции мероприя́тий.",
        },
        {
            "temp_def_id": "77df2f11-95a7-4d58-911c-b677582bc8f9",
            "def_sentence": "3 подхо́да по 15 подтягиваний",
        },
    ],
    "pronunciations": [
        {
            "entry_key": UUID("80fd5dad-fd73-51b6-95f9-67c87c49d461"),
            "text": "[pɐtˈxot]",
            "type": "ipa",
            "tags": [],
        }
    ],
    "verb_pairs": [],
}

In [21]:
fetcher = lfa()
processor = rlp()

In [22]:
words = ["жених", "квартира"]
results = fetcher.get(words)

In [30]:
zhenikh = {'word': 'жених',
  'entries': [{'language': {'code': 'ru', 'name': 'Russian'},
    'partOfSpeech': 'noun',
    'pronunciations': [{'type': 'ipa', 'text': '[ʐɨˈnʲix]', 'tags': []}],
    'forms': [{'word': 'жени́х',
      'tags': ['animate', 'canonical', 'masculine']},
     {'word': 'ženíx', 'tags': ['romanization']},
     {'word': 'жениха́', 'tags': ['genitive']},
     {'word': 'женихи́', 'tags': ['nominative', 'plural']},
     {'word': 'женихо́в', 'tags': ['genitive', 'plural']},
     {'word': 'женишо́к', 'tags': ['diminutive']},
     {'word': 'no-table-tags', 'tags': ['table-tags']},
     {'word': 'ru-noun-table', 'tags': ['inflection-template']},
     {'word': 'velar-stem', 'tags': ['class']},
     {'word': 'accent-b', 'tags': ['class']},
     {'word': 'жени́х', 'tags': ['nominative', 'singular']},
     {'word': 'женихи́', 'tags': ['nominative', 'plural']},
     {'word': 'жениха́', 'tags': ['genitive', 'singular']},
     {'word': 'женихо́в', 'tags': ['genitive', 'plural']},
     {'word': 'жениху́', 'tags': ['dative', 'singular']},
     {'word': 'жениха́м', 'tags': ['dative', 'plural']},
     {'word': 'жениха́', 'tags': ['accusative', 'singular']},
     {'word': 'женихо́в', 'tags': ['accusative', 'plural']},
     {'word': 'женихо́м', 'tags': ['instrumental', 'singular']},
     {'word': 'жениха́ми', 'tags': ['instrumental', 'plural']},
     {'word': 'женихе́', 'tags': ['prepositional', 'singular']},
     {'word': 'жениха́х', 'tags': ['plural', 'prepositional']},
     {'word': 'жени́хъ', 'tags': ['alternative']}],
    'senses': [{'definition': 'bridegroom; fiancé',
      'tags': [],
      'examples': [],
      'quotes': [{'text': 'Мно́гие из прия́тельниц тихо́нько поздравля́ли её с таки́м зави́дным женихо́м, но жени́х молча́л.',
        'reference': '1796, Николай Карамзин [Nikolay Karamzin], Юлия; English translation from (Please provide a date or year):'}],
      'synonyms': [],
      'antonyms': [],
      'subsenses': []}],
    'synonyms': [],
    'antonyms': ['неве́ста']}],
  'source': {'url': 'https://en.wiktionary.org/wiki/жених',
   'license': {'name': 'CC BY-SA 4.0',
    'url': 'https://creativecommons.org/licenses/by-sa/4.0/'}}}

In [14]:
db_lemma = ProcessedPayload(**podkhod)

In [15]:
db_lemma.lemmas

[LemmasRecord(clean_lemma='подход', accent_lemma='подхо́д', pos=5, entry_key=UUID('80fd5dad-fd73-51b6-95f9-67c87c49d461'))]

In [64]:
def _map_lemma(lemma_record: LemmasRecord):
    """_map_lemma _summary_

    Args:
        lemma_record (LemmasRecord): _description_

    Returns:
        _type_: _description_
    """
    mapped_lemma = Lemma(
        entry_key=lemma_record.entry_key,
        lem_text=lemma_record.clean_lemma,
        lem_canon=lemma_record.accent_lemma,
        pos=lemma_record.pos,
    )
    return mapped_lemma

In [65]:
def get_lemmas(
    db: Session,
    id: Optional[int] = None,
    entry_key: Optional[UUID] = None,
    clean_lemma: Optional[str] = None,
    accent_lemma: Optional[str] = None,
    pos: Optional[int] = None,
) -> List[Lemma]: # type: ignore
    """get_lemmas _summary_

    Args:
        db (Session): _description_
        id (Optional[int], optional): _description_. Defaults to None.
        entry_key (Optional[UUID], optional): _description_. Defaults to None.
        clean_lemma (Optional[str], optional): _description_. Defaults to None.
        accent_lemma (Optional[str], optional): _description_. Defaults to None.
        pos (Optional[int], optional): _description_. Defaults to None.

    Returns:
        List[Lemma]: _description_
    """
    # base statement
    stmt = select(Lemma)
    
    # dynamic chaining
    if clean_lemma is not None:
        stmt = stmt.where(Lemma.lem_text == clean_lemma)
    if accent_lemma is not None:
        stmt = stmt.where(Lemma.lem_canon == accent_lemma)
    if pos is not None:
        stmt = stmt.where(Lemma.pos == pos)
        
    return list(db.scalars(statement=stmt).all())

In [66]:
def create_lemma(db: Session, lemma_record: LemmasRecord):
    """create_lemma _summary_

    Args:
        db (Session): _description_
        lemma_record (LemmasRecord): _description_

    Returns:
        _type_: _description_
    """
    sql_lemma = _map_lemma(lemma_record=lemma_record)
    
    db.add(sql_lemma)
    db.flush()
    db.refresh(sql_lemma)

    return sql_lemma

In [71]:
db = SessionLocal()
word_lemma = "погода"
lem_result = get_lemmas(db=db, clean_lemma=word_lemma)
[(x.id, x.lem_canon) for x in lem_result]

[(7, 'пого́да')]